In [15]:
import pandas as pd
import re
import zipfile
from pathlib import Path

YEAR = 2024

HOME = Path.home()
current_path = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current_path, *current_path.parents] if (p / "Scripts").is_dir()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")

CRIME_DIR = HOME / "Desktop/Crime24"

lawyers_path = PROJECT_ROOT / "Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv"
acs_path = PROJECT_ROOT / "Data/Population Data/MSA Population/ACSDT1Y2024.B01003-Data.csv"

group_b_csv = PROJECT_ROOT / "Data/Proxies/Crime/nibrs_group_b_arrest_report_segment_2024.csv"
group_b_zip = HOME / "Downloads/group_b_arrest_report_segment_csv_1991_2024.zip"

output_path = PROJECT_ROOT / "Data/Proxies/Crime/Crime_Proxy_Normalized.csv"
state_folders = sorted(CRIME_DIR.glob(f"*-{YEAR}"))

In [17]:
def read_file(path):
    try:
        df = pd.read_csv(path, dtype=str, low_memory=False, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(path, dtype=str, low_memory=False, encoding="latin1")

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("ï»¿", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df


def find_file(folder, name):
    files = {path.name.lower(): path for path in folder.iterdir()}
    return files.get(name.lower())


def clean_key(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^A-Z0-9]", "", regex=True)
        .replace({"NA": "", "NAN": "", "NONE": "", "NULL": ""})
    )


def clean_msa(name):
    return re.sub(r"\s+", " ", str(name).replace(" Metro Area", "").strip())


def loose_msa(name):
    name = clean_msa(name)
    if "," not in name:
        return None

    city, states = name.rsplit(",", 1)
    city = re.split(r"[-–—]", city.strip())[0].upper()
    states = re.sub(r"\s+", "", states.upper())
    return f"{city}|{states}"


def to_area(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .replace("<NA>", pd.NA)
    )

In [18]:
agency_parts = []
ori_parts = []

for folder in state_folders:
    state = folder.name[:2]
    agencies_path = find_file(folder, "agencies.csv")

    if agencies_path is None:
        print("Missing agencies.csv:", folder.name)
        continue

    agencies = read_file(agencies_path)

    agency = agencies[["agency_id", "msa_name"]].copy()
    agency["state"] = state
    agency["agency_key"] = clean_key(agency["agency_id"])
    agency = agency[
        agency["agency_key"].ne("")
        & agency["msa_name"].notna()
        & agency["msa_name"].str.strip().ne("")
    ]
    agency_parts.append(agency[["state", "agency_key", "msa_name"]].drop_duplicates())

    ori = agencies[["msa_name", "ori", "legacy_ori", "covered_by_legacy_ori"]].melt(
        id_vars="msa_name",
        var_name="source",
        value_name="ori_value",
    )
    ori["ori_key"] = clean_key(ori["ori_value"])
    ori = ori[
        ori["ori_key"].ne("")
        & ori["msa_name"].notna()
        & ori["msa_name"].str.strip().ne("")
    ]
    ori_parts.append(ori[["ori_key", "msa_name", "source"]])

agency_map = pd.concat(agency_parts, ignore_index=True).drop_duplicates(
    ["state", "agency_key"]
)

ori_map = pd.concat(ori_parts, ignore_index=True)
ori_map["priority"] = ori_map["source"].map(
    {"ori": 1, "legacy_ori": 2, "covered_by_legacy_ori": 3}
)
ori_map = (
    ori_map.sort_values(["ori_key", "priority"])
    .drop_duplicates("ori_key")
    [["ori_key", "msa_name"]]
)

incident_parts = []
arrest_parts = []

for folder in state_folders:
    state = folder.name[:2]
    incident_path = find_file(folder, "NIBRS_incident.csv")
    arrestee_path = find_file(folder, "NIBRS_ARRESTEE.csv")

    if incident_path is None or arrestee_path is None:
        print("Missing NIBRS file:", folder.name)
        continue

    incidents = read_file(incident_path)
    incidents = incidents[pd.to_numeric(incidents["data_year"], errors="coerce").eq(YEAR)].copy()
    incidents["incident_key"] = clean_key(incidents["incident_id"])
    incidents["agency_key"] = clean_key(incidents["agency_id"])
    incidents["state"] = state

    incidents = incidents[
        incidents["incident_key"].ne("")
        & incidents["agency_key"].ne("")
    ]

    mapped_incidents = incidents.merge(
        agency_map,
        on=["state", "agency_key"],
        how="left",
    ).dropna(subset=["msa_name"])

    mapped_incidents["incident_uid"] = state + "_" + mapped_incidents["incident_key"]
    incident_parts.append(
        mapped_incidents[["msa_name", "incident_uid"]].drop_duplicates()
    )

    arrestees = read_file(arrestee_path)
    arrestees = arrestees[
        pd.to_numeric(arrestees["data_year"], errors="coerce").eq(YEAR)
    ].copy()

    if "arrestee_id" not in arrestees.columns:
        arrestees = arrestees.reset_index().rename(columns={"index": "arrestee_id"})

    arrestees["incident_key"] = clean_key(arrestees["incident_id"])
    arrestees["arrestee_key"] = clean_key(arrestees["arrestee_id"])
    arrestees = arrestees[arrestees["incident_key"].ne("")]

    incident_agency = incidents[
        ["incident_key", "agency_key"]
    ].drop_duplicates("incident_key")

    mapped_arrests = (
        arrestees.merge(incident_agency, on="incident_key", how="left")
        .assign(state=state)
        .merge(agency_map, on=["state", "agency_key"], how="left")
        .dropna(subset=["msa_name"])
    )

    mapped_arrests["arrest_uid"] = (
        "A_" + state + "_"
        + mapped_arrests["incident_key"]
        + "_"
        + mapped_arrests["arrestee_key"]
    )
    arrest_parts.append(
        mapped_arrests[["msa_name", "arrest_uid"]].drop_duplicates()
    )

group_a_incidents = (
    pd.concat(incident_parts, ignore_index=True)
    .drop_duplicates("incident_uid")
    .groupby("msa_name", as_index=False)
    .agg(Number_of_incidents=("incident_uid", "nunique"))
)

group_a_arrests = (
    pd.concat(arrest_parts, ignore_index=True)
    .drop_duplicates("arrest_uid")
    .groupby("msa_name", as_index=False)
    .agg(Number_of_arrests_GroupA=("arrest_uid", "nunique"))
)

In [20]:
if group_b_csv.exists():
    group_b = read_file(group_b_csv)
else:
    with zipfile.ZipFile(group_b_zip) as archive:
        member = next(
            name for name in archive.namelist()
            if name.lower().endswith(
                f"nibrs_group_b_arrest_report_segment_{YEAR}.csv"
            )
        )
        with archive.open(member) as source:
            group_b = pd.read_csv(
                source,
                dtype=str,
                low_memory=False,
                encoding="utf-8-sig",
            )
        group_b.columns = (
            group_b.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
        )

group_b_state = group_b["state_abb"].str.upper().replace({"NB": "NE"})

group_b = group_b[
    pd.to_numeric(group_b["year"], errors="coerce").eq(YEAR)
    & group_b_state.isin([folder.name[:2] for folder in state_folders])
].copy()

group_b["ori_key"] = clean_key(group_b["ori"])
group_b = group_b[group_b["ori_key"].ne("")].reset_index()

tx = group_b["arrest_transaction_incident_num"].fillna("").str.strip().str.upper()
seq = group_b["arrestee_sequence_number"].fillna("").str.strip().str.upper()
date = group_b["arrest_date"].fillna("").str.strip()
offense = group_b["ucr_arrest_offense_code"].fillna("").str.strip().str.upper()

group_b["arrest_uid"] = "B_" + group_b["ori_key"] + "_" + tx + "_" + seq

missing_key = tx.eq("") | seq.eq("")
group_b.loc[missing_key, "arrest_uid"] = (
    "B_FALLBACK_"
    + group_b.loc[missing_key, "ori_key"]
    + "_"
    + date.loc[missing_key]
    + "_"
    + offense.loc[missing_key]
    + "_"
    + group_b.loc[missing_key, "index"].astype(str)
)

group_b_arrests = (
    group_b.merge(ori_map, on="ori_key", how="left")
    .dropna(subset=["msa_name"])
    .drop_duplicates("arrest_uid")
    .groupby("msa_name", as_index=False)
    .agg(Number_of_arrests_GroupB=("arrest_uid", "nunique"))
)

acs = pd.read_csv(acs_path, dtype=str, low_memory=False, encoding="utf-8-sig")
acs.columns = acs.columns.astype(str).str.strip().str.replace("\ufeff", "", regex=False)

acs = acs[
    acs["NAME"].astype(str).str.contains("Metro Area", na=False)
].copy()

acs["AREA"] = to_area(acs["GEO_ID"].str.extract(r"(\d{5})$")[0])
acs["msa_clean"] = acs["NAME"].map(clean_msa)
acs["loose_key"] = acs["NAME"].map(loose_msa)

exact_area = (
    acs.dropna(subset=["AREA"])
    .drop_duplicates("msa_clean")
    .set_index("msa_clean")["AREA"]
)

loose_area = (
    acs.dropna(subset=["AREA", "loose_key"])
    .drop_duplicates("loose_key")
    .set_index("loose_key")["AREA"]
)


def add_area(df):
    df = df.copy()
    df["AREA"] = df["msa_name"].map(clean_msa).map(exact_area)
    missing = df["AREA"].isna()
    df.loc[missing, "AREA"] = (
        df.loc[missing, "msa_name"].map(loose_msa).map(loose_area)
    )
    return df


incidents_cbsa = (
    add_area(group_a_incidents)
    .dropna(subset=["AREA"])
    .groupby("AREA", as_index=False)["Number_of_incidents"]
    .sum()
)

arrests_a_cbsa = (
    add_area(group_a_arrests)
    .dropna(subset=["AREA"])
    .groupby("AREA", as_index=False)["Number_of_arrests_GroupA"]
    .sum()
)

arrests_b_cbsa = (
    add_area(group_b_arrests)
    .dropna(subset=["AREA"])
    .groupby("AREA", as_index=False)["Number_of_arrests_GroupB"]
    .sum()
)

In [21]:
lawyers = pd.read_csv(lawyers_path, low_memory=False)

lawyers["AREA"] = to_area(lawyers["CBSA"])
lawyers["Criminal_Defense"] = pd.to_numeric(
    lawyers["Criminal Defense_normalized_1overN_count"],
    errors="coerce",
)

lawyers = (
    lawyers[["AREA", "Criminal_Defense"]]
    .dropna()
    .groupby("AREA", as_index=False)["Criminal_Defense"]
    .max()
)

crime_proxy = (
    lawyers.merge(incidents_cbsa, on="AREA", how="left")
    .merge(arrests_a_cbsa, on="AREA", how="left")
    .merge(arrests_b_cbsa, on="AREA", how="left")
)

count_columns = [
    "Number_of_incidents",
    "Number_of_arrests_GroupA",
    "Number_of_arrests_GroupB",
]

crime_proxy[count_columns] = (
    crime_proxy[count_columns]
    .fillna(0)
    .astype("int64")
)

crime_proxy["Number_of_arrests"] = (
    crime_proxy["Number_of_arrests_GroupA"]
    + crime_proxy["Number_of_arrests_GroupB"]
)

zero_incident_areas = set(
    crime_proxy.loc[
        crime_proxy["Number_of_incidents"].eq(0),
        "AREA",
    ].astype(str)
)

assert zero_incident_areas == {"26140", "42700"}

crime_proxy = (
    crime_proxy.loc[crime_proxy["Number_of_incidents"].gt(0)]
    [["AREA", "Criminal_Defense", "Number_of_incidents", "Number_of_arrests"]]
    .sort_values("AREA")
    .reset_index(drop=True)
)

assert len(crime_proxy) == 380

output_path.parent.mkdir(parents=True, exist_ok=True)
crime_proxy.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(crime_proxy))
print("Incidents:", crime_proxy["Number_of_incidents"].sum())
print("Arrests:", crime_proxy["Number_of_arrests"].sum())

crime_proxy.head(20)

Saved: /Users/maxbelykh/Downloads/The-Urban-Legal-Mirage-main/Data/Proxies/Crime/Crime_Proxy_Normalized.csv
Rows: 380
Incidents: 10796544
Arrests: 5218944


,AREA,Criminal_Defense,Number_of_incidents,Number_of_arrests
0,10180,45.840873,7830,3232
1,10420,214.501154,29425,9647
2,10500,30.667857,4713,3191
3,10540,23.001190,2319,1731
4,10580,279.702165,20775,7096
5,10740,564.491400,67865,23147
6,10780,30.991667,9859,3536
7,10900,87.796429,13252,8979
8,11020,14.397619,2672,2567
9,11100,103.764683,15927,5390
